# Day 10: End-to-End Orchestration, Observability & Verification

This notebook simulates a complete E2E run of a synthetic patient from raw JSON -> Bronze -> Tokenization -> Silver OMOP -> DQ -> Gold metric.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, udf, date_add, datediff
from pyspark.sql.types import StringType
import hashlib
import json

spark = SparkSession.builder \
    .appName("Day10_E2E_Walkthrough") \
    .getOrCreate()

print("Spark session initialized.")

### Step 1: Raw to Bronze

In [ ]:
raw_json = [
    '{"patient_id": "123-45-6789", "event_date": "2023-05-01", "diagnosis": "J45", "heart_rate": 80}',
    '{"patient_id": "123-45-6789", "event_date": "2023-05-10", "diagnosis": "J45", "heart_rate": 85}', # Readmission
    '{"patient_id": null, "event_date": "2023-05-02", "diagnosis": "E11", "heart_rate": 70}' # Bad record
]
df_raw = spark.createDataFrame([(j,) for j in raw_json], ["raw_payload"])
bronze_df = spark.read.json(df_raw.rdd.map(lambda r: r.raw_payload))
print("Bronze Data:")
bronze_df.show()

### Step 2: Data Quality & DLQ

In [ ]:
good_df = bronze_df.filter(col("patient_id").isNotNull())
bad_df = bronze_df.filter(col("patient_id").isNull())
print("Good Data (Proceeds):")
good_df.show()
print("DLQ Data:")
bad_df.show()

### Step 3: De-identification (Tokenization)

In [ ]:
def mock_fpe(val): 
    return hashlib.sha256(val.encode()).hexdigest()[:11] if val else val
fpe_udf = udf(mock_fpe, StringType())

deid_df = good_df.withColumn("patient_id_fpe", fpe_udf(col("patient_id"))).drop("patient_id")
print("De-identified Data:")
deid_df.show()

### Step 4: Silver OMOP Transformation

In [ ]:
silver_df = deid_df.select(
    col("patient_id_fpe").alias("person_id"),
    col("event_date").alias("condition_start_date"),
    col("diagnosis").alias("condition_source_value")
)
print("Silver OMOP Data:")
silver_df.show()

### Step 5: Gold Mart (30-Day Readmission)

In [ ]:
silver_df.createOrReplaceTempView("silver_data")
gold_sql = """
SELECT 
    a.person_id,
    COUNT(*) as readmissions
FROM silver_data a
JOIN silver_data b 
    ON a.person_id = b.person_id
    AND b.condition_start_date > a.condition_start_date
    AND datediff(b.condition_start_date, a.condition_start_date) <= 30
GROUP BY a.person_id
"""
gold_df = spark.sql(gold_sql)
print("Gold Metric (Readmissions):")
gold_df.show()